In [ ]:
# rende config.py (in notebooks/) importabile anche dai notebook in sottocartelle
import sys, os
_cfg = os.getcwd()
while _cfg != os.path.dirname(_cfg):
    if os.path.exists(os.path.join(_cfg, 'config.py')):
        sys.path.insert(0, _cfg)
        break
    _cfg = os.path.dirname(_cfg)
from config import CONFESS_DATA, BC_DATA, ERA5_ROOT, POST_DATA, WORK_DIR, FIG_DIR, FIG_DIR_2025
import xarray as xr
import pandas as pd
import xskillscore as xs

import concurrent.futures
from concurrent.futures import ProcessPoolExecutor

import warnings
import logging

from pathlib import Path

import albedo_functions as af

In [ ]:
# Configura il logging
log_file = "process_log.txt"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(log_file, mode='w'),  # Write to file
#        logging.StreamHandler()  # Still see some output in Jupyter
    ]
)

In [ ]:
exp_ctrl = 'a1ua'
exp_sens = 'a52o'
var='tas'
era_var='2t'
COV_PATH = f'{POST_DATA}/'
LAI_PATH = f'{CONFESS_DATA}/'
SAVE_PATH = f'{WORK_DIR}/'
    
leads = [0, 1, 2, 3, 4]

In [ ]:
def main_script(var, era_var, exp_ctrl, exp_sens, era, save_path, lead,):
    """Main processing script for computing model acc against ERA5."""
    try:
        logging.info(f"Starting lead {lead}")
        # Load control experiment dataset
        ctrl_path = f"{save_path}/{exp_ctrl}/{exp_ctrl}_{var}_Amon_EC-Earth3_dcppA-hindcast_*_gr_1x1_ym_lead{lead}_rad.nc"
        dset_ctrl = xr.open_mfdataset(ctrl_path, concat_dim='member', combine='nested', chunks=-1)
        dset_ctrl['time'] = pd.to_datetime(dset_ctrl['time'].values).to_period('M').to_timestamp()
    
        # Load sensitivity experiment dataset
        sens_path = f"{save_path}/{exp_sens}/{exp_sens}_{var}_Amon_EC-Earth3_dcppA-hindcast_*_gr_1x1_ym_lead{lead}_rad.nc"
        dset_sens = xr.open_mfdataset(sens_path, concat_dim='member', combine='nested', chunks=-1)
        dset_sens['time'] = pd.to_datetime(dset_sens['time'].values).to_period('M').to_timestamp()
        
        # Load ERA5 reanalysis
        era['time'] = pd.to_datetime(era['time'].values).to_period('M').to_timestamp()
        era = era.sel(time=slice(dset_ctrl.time[0], dset_ctrl.time[-1]))
        
        #select time from 1999
        start_date = '1999-09-01'
        
        dset_ctrl = dset_ctrl.sel(time=slice(start_date, None))
        dset_sens = dset_sens.sel(time=slice(start_date, None))
        era = era.sel(time=slice(start_date, None))
        
        # Compute global mean
        dset_ctrl_global = af.global_average(dset_ctrl[var]).to_dataset(name = var)
        dset_sens_global = af.global_average(dset_sens[var]).to_dataset(name = var)
        era_global = af.global_average(era[era_var]).to_dataset(name = var)

        #compute land mean
        dset_ctrl_land = af.global_average(af.land_seas_mask(dset_ctrl[var], 'land')).to_dataset(name = var)
        dset_sens_land = af.global_average(af.land_seas_mask(dset_sens[var], 'land')).to_dataset(name = var)
        era_land = af.global_average(af.land_seas_mask(era[era_var], 'land')).to_dataset(name = var)

        #compute ocean mean
        dset_ctrl_ocean = af.global_average(af.land_seas_mask(dset_ctrl[var], 'ocean')).to_dataset(name = var)
        dset_sens_ocean = af.global_average(af.land_seas_mask(dset_sens[var], 'ocean')).to_dataset(name = var)
        era_ocean = af.global_average(af.land_seas_mask(era[era_var], 'ocean')).to_dataset(name = var)

        logging.info(f"Starting anomalie")
        # Calcolo anomalie
        dset_ctrl_anom = dset_ctrl_global - dset_ctrl_global.mean('time')
        dset_sens_anom = dset_sens_global - dset_sens_global.mean('time')
        
        dset_ctrl_land_anom = dset_ctrl_land - dset_ctrl_land.mean('time')
        dset_sens_land_anom = dset_sens_land - dset_sens_land.mean('time')
        
        dset_ctrl_ocean_anom = dset_ctrl_ocean - dset_ctrl_ocean.mean('time')
        dset_sens_ocean_anom = dset_sens_ocean - dset_sens_ocean.mean('time')
       
        era_anom = era_global - era_global.mean('time')
        era_land_anom = era_land - era_land.mean('time')
        era_ocean_anom = era_ocean - era_ocean.mean('time')    
        
        logging.info(f"Starting correlazioni")
        # Calcolo correlazioni
        em_corr_ctrl = xs.pearson_r(era_anom, dset_ctrl_anom.mean('member'), dim='time', skipna=True)
        corr_ctrl = xs.pearson_r(era_anom, dset_ctrl_anom,  dim='time', skipna=True)
        em_corr_sens = xs.pearson_r(era_anom, dset_sens_anom.mean('member'), dim='time', skipna=True)
        corr_sens = xs.pearson_r(era_anom, dset_sens_anom,  dim='time', skipna=True)

        em_corr_land_ctrl = xs.pearson_r(era_land_anom, dset_ctrl_land_anom.mean('member'), dim='time', skipna=True)
        corr_ctrl_land = xs.pearson_r(era_land_anom, dset_ctrl_land_anom,  dim='time', skipna=True)
        em_corr_land_sens = xs.pearson_r(era_land_anom, dset_sens_land_anom.mean('member'), dim='time', skipna=True)
        corr_sens_land = xs.pearson_r(era_land_anom, dset_sens_land_anom,  dim='time', skipna=True)

        em_corr_ocean_ctrl = xs.pearson_r(era_ocean_anom, dset_ctrl_ocean_anom.mean('member'), dim='time', skipna=True)
        corr_ctrl_ocean = xs.pearson_r(era_ocean_anom, dset_ctrl_ocean_anom,  dim='time', skipna=True)
        em_corr_ocean_sens = xs.pearson_r(era_ocean_anom, dset_sens_ocean_anom.mean('member'), dim='time', skipna=True)
        corr_sens_ocean = xs.pearson_r(era_ocean_anom, dset_sens_ocean_anom,  dim='time', skipna=True)

        # Save results
        ctrl_em_outfile = f'{save_path}/{exp_ctrl}_{var}_lead_{lead}_1x1_global_acc_em_1999.nc'
        ctrl_outfile = f'{save_path}/{exp_ctrl}_{var}_lead_{lead}_1x1_global_acc_1999.nc'
        sens_em_outfile = f'{save_path}/{exp_sens}_{var}_lead_{lead}_1x1_global_acc_em_1999.nc'
        sens_outfile = f'{save_path}/{exp_sens}_{var}_lead_{lead}_1x1_global_acc_1999.nc'

        ctrl_em_land_outfile = f'{save_path}/{exp_ctrl}_{var}_lead_{lead}_1x1_global_acc_em_land_1999.nc'
        ctrl_land_outfile = f'{save_path}/{exp_ctrl}_{var}_lead_{lead}_1x1_global_acc_land_1999.nc'
        sens_em_land_outfile = f'{save_path}/{exp_sens}_{var}_lead_{lead}_1x1_global_acc_em_land_1999.nc'
        sens_land_outfile = f'{save_path}/{exp_sens}_{var}_lead_{lead}_1x1_global_acc_land_1999.nc'

        ctrl_em_ocean_outfile = f'{save_path}/{exp_ctrl}_{var}_lead_{lead}_1x1_global_acc_em_ocean_1999.nc'
        ctrl_ocean_outfile = f'{save_path}/{exp_ctrl}_{var}_lead_{lead}_1x1_global_acc_ocean_1999.nc'
        sens_em_ocean_outfile = f'{save_path}/{exp_sens}_{var}_lead_{lead}_1x1_global_acc_em_ocean_1999.nc'
        sens_ocean_outfile = f'{save_path}/{exp_sens}_{var}_lead_{lead}_1x1_global_acc_ocean_1999.nc'

        em_corr_ctrl.to_netcdf(ctrl_em_outfile)
        corr_ctrl.to_netcdf(ctrl_outfile)
        em_corr_sens.to_netcdf(sens_em_outfile)
        corr_sens.to_netcdf(sens_outfile)

        em_corr_land_ctrl.to_netcdf(ctrl_em_land_outfile)
        corr_ctrl_land.to_netcdf(ctrl_land_outfile)
        em_corr_land_sens.to_netcdf(sens_em_land_outfile)
        corr_sens_land.to_netcdf(sens_land_outfile)
        
        em_corr_ocean_ctrl.to_netcdf(ctrl_em_ocean_outfile)
        corr_ctrl_ocean.to_netcdf(ctrl_ocean_outfile)
        em_corr_ocean_sens.to_netcdf(sens_em_ocean_outfile)
        corr_sens_ocean.to_netcdf(sens_ocean_outfile) 
        

        # --- significativita' bootstrap della differenza di ACC (media d'ensemble) ---
        sig_global = af.bootstrap_acc_significance(dset_sens_anom[var], dset_ctrl_anom[var], era_anom[var])
        sig_land   = af.bootstrap_acc_significance(dset_sens_land_anom[var], dset_ctrl_land_anom[var], era_land_anom[var])
        sig_ocean  = af.bootstrap_acc_significance(dset_sens_ocean_anom[var], dset_ctrl_ocean_anom[var], era_ocean_anom[var])

        xr.DataArray(sig_global, name='sig').to_netcdf(f'{save_path}/{exp_sens}-{exp_ctrl}_{var}_lead_{lead}_1x1_global_accsig_1999.nc')
        xr.DataArray(sig_land,  name='sig').to_netcdf(f'{save_path}/{exp_sens}-{exp_ctrl}_{var}_lead_{lead}_1x1_global_accsig_land_1999.nc')
        xr.DataArray(sig_ocean, name='sig').to_netcdf(f'{save_path}/{exp_sens}-{exp_ctrl}_{var}_lead_{lead}_1x1_global_accsig_ocean_1999.nc')

        logging.info(f"Bias files written successfully for lead {lead}")
    
    except Exception as e:
        logging.exception(f"Error occurred for lead {lead}: {e}")

In [ ]:
%%time
# Load ERA5 reanalysis
era_path = f"{WORK_DIR}/ERA5_{era_var}_1x1_year.nc"
era = xr.open_dataset(era_path, chunks=-1).load()
futures=[]
with concurrent.futures.ProcessPoolExecutor(max_workers=2) as executor:
    futures = [executor.submit(main_script, var, era_var, exp_ctrl, exp_sens, era, SAVE_PATH, lead) for lead in leads]

    # Wait for all tasks to complete
    # progresso: stampa ogni task man mano che termina
    for _i, _f in enumerate(concurrent.futures.as_completed(futures), 1):
        _f.result()
        print(f"  completato {_i}/{len(futures)}", flush=True)

In [ ]:
# ============================================================================
# ACC per COMBINAZIONI di lead year (range contigui y1-y2), legge i file
# multi-anno ..._lead_{y1}-{y2}_1x1_ensemble_rad.nc (come 04-BIAS/process_lead_years)
# e calcola la stessa ACC di media d'area di main_script.
# ============================================================================
def _acc_region(ctrl_da, sens_da, era_da, mask):
    """ACC (per membro + media d'ensemble) + significativita' bootstrap per una regione."""
    if mask is None:
        cg = af.global_average(ctrl_da); sg = af.global_average(sens_da); eg = af.global_average(era_da)
    else:
        cg = af.global_average(af.land_seas_mask(ctrl_da, mask))
        sg = af.global_average(af.land_seas_mask(sens_da, mask))
        eg = af.global_average(af.land_seas_mask(era_da, mask))
    ca = cg - cg.mean('time'); sa = sg - sg.mean('time'); ea = eg - eg.mean('time')
    corr_c = xs.pearson_r(ea, ca, dim='time', skipna=True)
    corr_s = xs.pearson_r(ea, sa, dim='time', skipna=True)
    em_c = xs.pearson_r(ea, ca.mean('member'), dim='time', skipna=True)
    em_s = xs.pearson_r(ea, sa.mean('member'), dim='time', skipna=True)
    sig = af.bootstrap_acc_significance(sa, ca, ea)
    return corr_c, corr_s, em_c, em_s, sig


def main_script_ly(var, era_var, exp_ctrl, exp_sens, data_path, obs_path, save_path, y1, y2):
    """ACC di media d'area per il range di lead year y1-y2 (media contigua)."""
    lead = f"{y1}-{y2}"
    lead_number = y2 - y1 + 1
    try:
        logging.info(f"Starting ACC lead years {lead}")
        ctrl = xr.open_dataset(f"{data_path}/{exp_ctrl}/1x1/{var}/{exp_ctrl}_{var}_Amon_EC-Earth3_dcppA-hindcast_lead_{lead}_1x1_ensemble_rad.nc")
        sens = xr.open_dataset(f"{data_path}/{exp_sens}/1x1/{var}/{exp_sens}_{var}_Amon_EC-Earth3_dcppA-hindcast_lead_{lead}_1x1_ensemble_rad.nc")
        ctrl['time'] = pd.to_datetime(ctrl['time'].values).normalize().to_period('M').start_time
        sens['time'] = pd.to_datetime(sens['time'].values).normalize().to_period('M').start_time

        era = xr.open_dataset(f"{obs_path}/ERA5_{era_var}_1x1_{lead_number}year.nc")
        if lead_number in [2, 4]:
            era['time'] = pd.to_datetime(era['time'].values) - pd.DateOffset(months=1)
        era['time'] = pd.to_datetime(era['time'].values).normalize().to_period('M').start_time
        era = era.rename({era_var: var})
        era = era.sel(time=slice(ctrl.time[0], ctrl.time[-1]))

        start_date = '1999-09-01'
        ctrl = ctrl.sel(time=slice(start_date, None))
        sens = sens.sel(time=slice(start_date, None))
        era = era.sel(time=slice(start_date, None))

        for mask, suff in [(None, ''), ('land', '_land'), ('ocean', '_ocean')]:
            corr_c, corr_s, em_c, em_s, sig = _acc_region(ctrl[var], sens[var], era[var], mask)
            corr_c.to_dataset(name=var).to_netcdf(f'{save_path}/{exp_ctrl}_{var}_leadyears_{lead}_1x1_global_acc{suff}_1999.nc')
            corr_s.to_dataset(name=var).to_netcdf(f'{save_path}/{exp_sens}_{var}_leadyears_{lead}_1x1_global_acc{suff}_1999.nc')
            em_c.to_dataset(name=var).to_netcdf(f'{save_path}/{exp_ctrl}_{var}_leadyears_{lead}_1x1_global_acc_em{suff}_1999.nc')
            em_s.to_dataset(name=var).to_netcdf(f'{save_path}/{exp_sens}_{var}_leadyears_{lead}_1x1_global_acc_em{suff}_1999.nc')
            xr.DataArray(sig, name='sig').to_netcdf(f'{save_path}/{exp_sens}-{exp_ctrl}_{var}_leadyears_{lead}_1x1_global_accsig{suff}_1999.nc')

        logging.info(f"ACC lead years {lead} salvato")
    except Exception as e:
        logging.exception(f"Errore ACC lead years {lead}: {e}")


In [ ]:
%%time
# calcola l'ACC per tutti i range contigui y1-y2 (0-1, 0-2, ... 3-4)
from concurrent.futures import as_completed

with concurrent.futures.ProcessPoolExecutor(max_workers=2) as executor:
    futures = {}
    for y1 in range(5):
        for y2 in range(y1 + 1, 5):
            fut = executor.submit(main_script_ly, var, era_var, exp_ctrl, exp_sens,
                                  str(POST_DATA), str(WORK_DIR), SAVE_PATH, y1, y2)
            futures[fut] = f"{y1}-{y2}"
    for _i, _f in enumerate(as_completed(futures), 1):
        _f.result()
        print(f"  [{futures[_f]}] completato {_i}/{len(futures)}", flush=True)
